In [1]:
import pandas as pd
import numpy as np

In [3]:
# --------------------------------------------------
# 1. Load daily wind data
# --------------------------------------------------

# path
wind = pd.read_csv(
    "../Capstone/raw data/dim_wind_clean.csv", 
    parse_dates=["Date/Time"],
    low_memory=False
)

wind.columns = wind.columns.str.strip().str.replace(" ", "_")

# Rename key-columns
wind = wind.rename(columns={
    "Climate_ID": "ClimateID",
    "Spd_of_Max_Gust_(km/h)": "MaxGust_kmh"
})

In [6]:
# Ensure consistent names
wind = wind.rename(columns={
    "Spd of Max Gust (km/h)": "MaxGust_kmh"
})

# Remove lines with no gust value.
wind = wind.dropna(subset=["MaxGust_kmh"])

In [13]:
# --------------------------------------------------
# 2. Calculate P95 per station (ClimateID)
# --------------------------------------------------

p95_thresholds = (
    wind
    .groupby("ClimateID")["MaxGust_kmh"]
    .quantile(0.95)
    .reset_index()
    .rename(columns={"MaxGust_kmh": "P95_Threshold"})
)

In [14]:
# --------------------------------------------------
# 3. Merge thresholds back to daily data
# --------------------------------------------------

wind_p95 = wind.merge(
    p95_thresholds,
    on="ClimateID",
    how="left"
)

In [15]:
# --------------------------------------------------
# 4. Filter Extreme Wind Events
# --------------------------------------------------

extreme_wind = wind_p95[
    wind_p95["MaxGust_kmh"] >= wind_p95["P95_Threshold"]
].copy()

In [16]:
# --------------------------------------------------
# 5. Build Fact_Extreme_Events
# --------------------------------------------------

fact_extreme_events = extreme_wind[
    ["ClimateID", "Date/Time", "MaxGust_kmh", "P95_Threshold"]
].copy()

fact_extreme_events["EventType"] = "Extreme Wind"
fact_extreme_events["ExtremeFlag"] = 1

# Reordenar colunas
fact_extreme_events = fact_extreme_events[
    [
        "ClimateID",
        "Date/Time",
        "EventType",
        "MaxGust_kmh",
        "P95_Threshold",
        "ExtremeFlag"
    ]
]

In [17]:
# --------------------------------------------------
# 6. Sanity checks
# --------------------------------------------------

print("Number of extreme events:", len(fact_extreme_events))
print("\nExample per station:")
print(fact_extreme_events.groupby("ClimateID").size().head())

# Confirm ground rule
assert (
    fact_extreme_events["MaxGust_kmh"]
    >= fact_extreme_events["P95_Threshold"]
).all(), "There are events below P95!"

Number of extreme events: 25045

Example per station:
ClimateID
8100467    230
8100506    105
8100885    250
8100989    221
8101201    124
dtype: int64


In [18]:
# --------------------------------------------------
# 7. Export
# --------------------------------------------------

fact_extreme_events.to_csv(
    "../Capstone/clean data/Fact_Extreme_Events.csv",
    index=False
)